<a href="https://colab.research.google.com/github/carrilloluis25/ECE_551_problems_ML/blob/main/notebooks/fast_ddpm_viewer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fast-DDPM image viewer

View what Fast-DDPM does to a low-dose CT slice: **input | generated | ground truth | error**, with PSNR and SSIM.

Two modes:
- **A. Run the model on your own images** (inference with the authors' pretrained LDFDCT checkpoint).
- **B. View results produced on the cluster** (upload the generated `*_pt.png` and `*_gt.png` files).

Runs in two places; the first cell detects which:
- **Google Colab:** Runtime > Change runtime type > **GPU** (a T4 is enough). The first cell downloads the code and a 1.7 GB checkpoint (about a minute), and you upload images.
- **Easley JupyterHub:** start a server on a GPU partition with the `fastddpm` kernel. It uses the repo's `fast-ddpm` submodule and the checkpoint and test set already on scratch, so nothing is downloaded or uploaded. Set `CKPT` in the first cell to view one of your own training checkpoints.

This notebook is inference only; training needs the cluster.

In [ ]:
# 1. Get the code and the pretrained checkpoint
import os, sys, glob, zipfile, argparse, subprocess
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    CODE_DIR, CKPT = "Fast-DDPM", "ckpt_LDFDCT.pth"
    if not os.path.isdir(CODE_DIR):
        subprocess.run(["git", "clone", "-q", "https://github.com/mirthAI/Fast-DDPM.git"], check=True)
    subprocess.run(["wget", "-q", "-c", "https://huggingface.co/SebastianJiang/FastDDPM/resolve/main/ckpt_LDFDCT.pth",
                    "-O", CKPT], check=True)
else:
    # Easley: paths from docs/hpc_quickstart.md. Point CKPT at <exp>/logs/<doc>/ckpt_<step>.pth for your own run.
    CODE_DIR = os.path.expanduser("~/ECE_551_problems_ML/fast-ddpm")
    CKPT = os.path.expanduser("~/carc-scratch/fastddpm-exp/logs/ldfdct_smoke_test/ckpt_1.pth")
    assert os.path.isdir(CODE_DIR) and os.path.isfile(CKPT), f"check CODE_DIR={CODE_DIR} and CKPT={CKPT}"

sys.path.insert(0, CODE_DIR)
import torch
assert torch.cuda.is_available(), "No GPU: in Colab use Runtime > Change runtime type > GPU; on Easley start the server on a GPU partition"
print("torch", torch.__version__, "|", torch.cuda.get_device_name(0), "|", "Colab" if IN_COLAB else "Easley", "|", CKPT)

In [ ]:
# 2. Build the model and load the pretrained weights
import yaml, numpy as np, matplotlib.pyplot as plt
from PIL import Image
from skimage.metrics import structural_similarity as ssim
from models.diffusion import Model
from functions.denoising import sg_generalized_steps

def to_ns(d):
    ns = argparse.Namespace()
    for k, v in d.items():
        setattr(ns, k, to_ns(v) if isinstance(v, dict) else v)
    return ns

config = to_ns(yaml.safe_load(open(f"{CODE_DIR}/configs/ldfd_linear.yml")))
device = torch.device("cuda")

model = Model(config)
states = torch.load(CKPT, map_location="cpu", weights_only=False)
# checkpoint = [model, optimizer, epoch, step, EMA]; keys saved from DataParallel start with "module."
model.load_state_dict({k.replace("module.", "", 1): v for k, v in states[0].items()}, strict=True)
# The authors sample with the EMA (smoothed) weights, so copy them in, as EMAHelper.ema() does.
ema = states[-1]
n_params = sum(1 for _ in model.named_parameters())
n_ema = 0
for name, p in model.named_parameters():
    if name in ema:
        p.data.copy_(ema[name]); n_ema += 1
print(f"EMA weights applied to {n_ema}/{n_params} parameters; trained step {states[3]}")
model.to(device).eval()

betas = torch.linspace(config.diffusion.beta_start, config.diffusion.beta_end,
                       config.diffusion.num_diffusion_timesteps, dtype=torch.float64).float().to(device)

In [ ]:
# 3. Helpers: run the model, score, and display
@torch.no_grad()
def denoise(pil_img, timesteps=10, seed=0):
    size = config.data.image_size
    a = np.asarray(pil_img.convert("L").resize((size, size)), dtype=np.float32) / 255.0
    x_img = torch.from_numpy(a)[None, None].to(device) * 2 - 1          # condition, scaled to [-1, 1]
    torch.manual_seed(seed)
    x = torch.randn(1, 1, size, size, device=device)                     # start from pure noise
    skip = 1000 // timesteps
    seq = list(range(-1, 1000, skip)); seq[0] = 0                        # same steps as the repo's uniform scheduler
    xs, _ = sg_generalized_steps(x, x_img, seq, model, betas, eta=0.0)   # deterministic DDIM
    return ((xs[-1] + 1) / 2).clamp(0, 1)[0, 0].float().cpu().numpy(), a

def psnr(a, b):
    mse = np.mean((a * 255 - b * 255) ** 2)
    return float("inf") if mse == 0 else 20 * np.log10(255.0 / np.sqrt(mse))

def show(rows):
    # rows: list of (title, input or None, output, ground truth or None), arrays in [0, 1]
    cols = 4
    fig, axes = plt.subplots(len(rows), cols, figsize=(3.4 * cols, 3.4 * len(rows)), squeeze=False)
    for r, (title, inp, out, gt) in enumerate(rows):
        panels = [("input (low dose)", inp), ("generated", out), ("ground truth", gt),
                  ("|generated - truth|", None if gt is None else np.abs(out - gt))]
        for c, (name, img) in enumerate(panels):
            ax = axes[r][c]; ax.axis("off")
            if img is None:
                continue
            ax.imshow(img, cmap="hot" if c == 3 else "gray", vmin=0, vmax=0.3 if c == 3 else 1)
            ax.set_title(name, fontsize=10)
        if gt is not None:
            axes[r][1].set_xlabel(f"PSNR {psnr(out, gt):.2f} dB   SSIM {ssim(gt, out, data_range=1.0):.4f}")
            axes[r][1].xaxis.set_label_position("top")
        axes[r][0].text(0, -0.04, title, transform=axes[r][0].transAxes, fontsize=9, va="top")
    plt.tight_layout(); plt.show()

## A. Run the model on your own images
In Colab, upload one or more PNG/JPG images, or a `.zip` of them. On Easley, set `INPUT_DIR` to a folder instead (the default is the LDFDCT test set). Naming:
- `NAME_ld.png` = low-dose input, with an optional `NAME_fd.png` = full-dose ground truth (then you get PSNR/SSIM and an error map).
- Any other image is treated as an input with no ground truth.

Images are converted to grayscale and resized to 256 x 256. This model was trained on low-dose CT slices, so other kinds of images will not mean much.

In [ ]:
if IN_COLAB:
    from google.colab import files
    INPUT_DIR = "inputs"
    os.makedirs(INPUT_DIR, exist_ok=True)
    for name, data in files.upload().items():
        open(f"{INPUT_DIR}/{name}", "wb").write(data)
        if name.lower().endswith(".zip"):
            zipfile.ZipFile(f"{INPUT_DIR}/{name}").extractall(INPUT_DIR)
else:
    INPUT_DIR = os.path.expanduser("~/carc-scratch/fastddpm-data/data/LD_FD_CT_test")
print("reading images from", INPUT_DIR)

In [ ]:
TIMESTEPS = 10      # 10 is what the model was trained for; 5 and 2 are subsets of its trained steps
MAX_IMAGES = 8

paths = sorted(p for p in glob.glob(f"{INPUT_DIR}/**/*", recursive=True)
               if p.lower().endswith((".png", ".jpg", ".jpeg")) and not p.endswith("_fd.png"))
paths = paths[::max(1, len(paths) // MAX_IMAGES)][:MAX_IMAGES]   # spread across the folder, not 8 neighbouring slices
rows = []
for p in paths:
    out, inp = denoise(Image.open(p), timesteps=TIMESTEPS)
    gt = None
    if p.endswith("_ld.png") and os.path.exists(p.replace("_ld.png", "_fd.png")):
        g = Image.open(p.replace("_ld.png", "_fd.png")).convert("L").resize((config.data.image_size,) * 2)
        gt = np.asarray(g, dtype=np.float32) / 255.0
    rows.append((os.path.basename(p), inp, out, gt))
print(f"{len(rows)} image(s), {TIMESTEPS} sampling steps")
show(rows)

## B. View results produced on the cluster
The cluster's evaluation writes `<ckpt>_<id>_pt.png` (generated) and `<ckpt>_<id>_gt.png` (ground truth) into `exp/image_samples/<run>/images_fid/`. In Colab, zip a few of them and upload the zip; on Easley, set `RESULTS_DIR` to that `images_fid/` folder. Then run the next two cells. No GPU work happens in this mode.

In [ ]:
if IN_COLAB:
    RESULTS_DIR = "cluster_results"
    os.makedirs(RESULTS_DIR, exist_ok=True)
    for name, data in files.upload().items():
        open(f"{RESULTS_DIR}/{name}", "wb").write(data)
        if name.lower().endswith(".zip"):
            zipfile.ZipFile(f"{RESULTS_DIR}/{name}").extractall(RESULTS_DIR)
else:
    # Narrow this to one run, e.g. .../image_samples/<run>/images_fid, so runs are not mixed.
    RESULTS_DIR = os.path.expanduser("~/carc-scratch/fastddpm-exp/image_samples")
print("reading results from", RESULTS_DIR)

In [ ]:
MAX_IMAGES = 8
pts = sorted(glob.glob(f"{RESULTS_DIR}/**/*_pt.png", recursive=True))[:MAX_IMAGES]
rows = []
for p in pts:
    gtp = p.replace("_pt.png", "_gt.png")
    out = np.asarray(Image.open(p).convert("L"), dtype=np.float32) / 255.0
    gt = np.asarray(Image.open(gtp).convert("L"), dtype=np.float32) / 255.0 if os.path.exists(gtp) else None
    rows.append((os.path.relpath(p, RESULTS_DIR), None, out, gt))
print(f"{len(rows)} result(s)")
show(rows)

## Making a small sample on the cluster (Colab only)
To get a handful of low-dose / full-dose pairs to upload in mode A, run this on Easley (it picks about 8 pairs spread across the test set). Check the dataset's usage terms before sharing the images publicly.

```bash
cd ~/carc-scratch/fastddpm-data/data/LD_FD_CT_test
find . -name "*_ld.png" | sort | awk 'NR%400==1' | head -8 | while read f; do echo "$f"; echo "${f%_ld.png}_fd.png"; done | zip ~/sample_pairs.zip -@
```

Then copy `~/sample_pairs.zip` to your computer with `scp`.

**Notes.** Written against the Fast-DDPM source (model, EMA weights and sampler are the repo's own); not yet run in Colab, so report any error. For several images the model runs one at a time, a fraction of a second each on a T4.